In [3]:
import pandas as pd
import numpy as np
import faiss
import ollama
from sentence_transformers import SentenceTransformer

/Users/ayhan/Desktop/DS_Bootcamp/Capstone_Martin_Ayhan/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df = pd.read_csv("../data/BMW/clean_reviews.csv")
df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW


In [6]:
#Embedding Modell laden
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

In [8]:
#FAISS Index
index = faiss.read_index("../models/review_index.faiss")

In [9]:
#Embeddings laden
embeddings = np.load("../models/review_embeddings.npy")
print(embeddings.shape)

(10509, 384)


In [10]:
#Semantic Retrieval Funktion
def retrieve_reviews(query, k=20):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, k)
    results =df.iloc[indices[0]]
    return results

In [12]:
#Retrieval testen
results = retrieve_reviews("charging problems BMW app")
results[["content"]]

,content
3577,"time to time i get this error in this app: ""ch..."
2618,app is missing the possibility to adjust the c...
2947,very disappointing how buggy and unintuitive t...
501,the charging features through the app don't wo...
1590,the app does not now inform me when charging t...
2446,app no longer working. mileage incorrect. says...
433,hit and miss. my car has been charging for thr...
3096,charging fails to work for char.gy chargers. h...
5050,works great with car app works well with my 20...
2891,they have moved the bmw charging app to my bmw...


In [13]:
#Reviews in Kontext umwandeln
def build_context(reviews):
    context ="\n".join(reviews["content"].tolist())
    return context


In [27]:
import ollama

def generate_startup_idea(problem_query):

    # Retrieve reviews
    reviews = retrieve_reviews(problem_query)

    if reviews.empty:
        return "No reviews found."

    # Build context (limit length)
    context = build_context(reviews[:50])

    prompt = f"""
You are an expert startup founder and venture capitalist.

Analyze the following customer reviews to discover unmet needs.

Reviews:
{context}

IMPORTANT:
The startup idea MUST be realistic and feasible.
Consider the following constraints:
- The solution must be technically feasible with today's technology.
- It must not depend on private or restricted APIs from large companies (e.g., BMW internal systems).
- It should be possible for a startup to build an MVP within 6–12 months.
- The business model must be viable for an independent startup.
- Avoid ideas that require cooperation from the original company unless it is realistic.

Generate ONE startup idea using this structure:

Startup Name:
Problem:
Evidence from Reviews:
Solution / Product Idea:
Target Market:
Value Proposition:
Business Model:
Minimum Viable Product:
Go-To-Market Strategy:
Competitive Advantage:
Risks:
Strategic Recommendations:
"""

    try:
        response = ollama.chat(
            model="llama3",
            messages=[{"role": "user", "content": prompt}]
        )

        result = response["message"]["content"]
        return result

    except Exception as e:
        return f"LLM Error: {e}"

In [28]:
#Funktion testen
print(generate_startup_idea("BMW app charging problems"))

Here's a startup idea based on the customer reviews:

**Startup Name:** ChargeKeeper

**Problem:** The current BMW charging app is unreliable, buggy, and lacks essential features, leading to frustration and inconvenience for electric vehicle (EV) owners.

**Evidence from Reviews:**

* 11 out of 14 reviewers experienced issues with the app's reliability, such as incorrect mileage readings, inconsistent charging data, or failure to charge according to scheduled plans.
* 9 reviewers mentioned specific feature requests, including adjusting charge percentages, setting maximum allowed current (ampère), and receiving notifications when charging is complete.

**Solution/Product Idea:** ChargeKeeper is a standalone EV charging app designed specifically for BMW owners. It will address the issues and feature gaps identified in the reviews by providing a reliable, user-friendly, and feature-rich solution.

**Target Market:** Electric vehicle owners with BMW models (e.g., i3, i4, IX3).

**Value Pro

In [29]:
#Mehrere Startup Ideen generieren
def generate_multiple_startups(problem_list):
    for problem in problem_list:
        print("\n=================")
        print("Problem:", problem)
        print("=================\n")
        generate_startup_idea(problem)

In [30]:
#Test Mehrere Startup Ideen
problems = ["charging problems BMW app ", "BMW app connection issues", "BMW app login problems"]
generate_multiple_startups(problems)


Problem: charging problems BMW app 


Problem: BMW app connection issues


Problem: BMW app login problems



In [31]:
#Automatische Problem Discovery
def discover_problems():
    sample_reviews = df.sample(50)
    return sample_reviews["content"]

In [32]:
#AI Opportunity Generator
def startup_opportunity_engine():
    reviews = discover_problems()
    context = "\n".join(reviews.tolist())
    prompt = f"""
Analyze these user reviews.
Find the biggest product problems.
Generate startup opportunities.
Reviews: {context}
"""
    response = ollama.chat(model="llama3", messages=[{"role":"user", "content": prompt}])
    print(response["message"]["content"])

In [33]:
#Test
startup_opportunity_engine()

**Biggest Product Problems:**

1. **Unreliable Vehicle Finder**: Many users reported that the vehicle finder feature stopped working, making it difficult to locate their cars.
2. **Inconsistent Data Updates**: Some users experienced issues with data updates not syncing correctly, causing frustration and a lack of trust in the app's functionality.
3. **Limited Compatibility**: The app struggled with compatibility issues on certain devices or operating systems, leading to a poor user experience.
4. **Login Issues**: Users reported frequent login requirements, making the app inconvenient to use.
5. **Slow Response Time**: Some users experienced slow response times when interacting with the app, which can be frustrating and lead to abandonment.

**Startup Opportunities:**

1. **Improved Vehicle Finder**: Develop a more reliable vehicle finder feature that accurately locates cars in real-time.
2. **Enhanced Data Syncing**: Implement a seamless data syncing mechanism to ensure users always h